In [1]:
!pip install diffusers transformers accelerate safetensors peft pytorch-fid lpips clip-score open_clip_torch pillow torch torchvision

  Using cached torch-2.10.0-cp314-cp314-macosx_14_0_arm64.whl.metadata (31 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 6.7 MB/s  0:00:006.7 MB/s eta 0:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 616.3/616.3 kB 4.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 5.7 MB/s  0:00:00m 5.8 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 6.7 MB/s  0:00:01.8 MB/s eta 0:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 5.2 MB/s  0:00:005.3 MB/s eta 0:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 6.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 6.5 MB/s  0:00:00m 6.9 MB/s eta 0:00:01
Using cached torch-2.10.0-cp314-cp314-macosx_14_0_arm64.whl (79.5 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 8.1 MB/s  0:00:00m 8.3 MB/s eta 0:00:01
  Attempting uninstall: torch
    Found existing installation: torch 2.12.0a0+gitc3d42f8
    Uninstalling torch-2.12.0a0+gi

In [1]:
import torch
import torch.nn.functional as F
from import StableDiffusionPipeline, DDPMScheduler, AutoencoderKL, UNet2DConditionModel
from transformers import CLIPTextModel, CLIPTokenizer
from peft import LoraConfig, get_peft_model, PeftModel
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from pathlib import Path
from tqdm import tqdm
import os
import matplotlib.pyplot as plt

/Users/bogdan/Developer/environments/venv/torch-build/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DEVICE = "mps"
DTYPE = torch.float16

MODEL_ID = "runwayml/stable-diffusion-v1-5"
OUTPUT_DIR = "./lora_model"
INSTANCE_DIR = "./my_photos"
CLASS_DIR = "./class_images"
GENERATED_DIR = "./generated_images"
VAL_DIR = "./val_images"

os.makedirs(INSTANCE_DIR, exist_ok=True)
os.makedirs(CLASS_DIR, exist_ok=True)
os.makedirs(GENERATED_DIR, exist_ok=True)
os.makedirs(VAL_DIR, exist_ok=True)

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

INSTANCE_PROMPT = "a photo of sks man, face portrait, looking at camera"
CLASS_PROMPT = "a photo of man, face portrait, looking at camera"
VAL_PROMPT = "a portrait photo of sks man, high quality, realism, face close-up"

In [7]:
NUM_CLASS_IMAGES = 30

existing_class = [p for p in Path(CLASS_DIR).iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS]
num_to_generate = NUM_CLASS_IMAGES - len(existing_class)

if num_to_generate > 0:
    pipe_tmp = StableDiffusionPipeline.from_pretrained(MODEL_ID, torch_dtype=DTYPE)
    pipe_tmp = pipe_tmp.to(DEVICE)
    pipe_tmp.safety_checker = None

    print(f"Generating {num_to_generate} class images...")
    for i in range(num_to_generate):
        image = pipe_tmp(CLASS_PROMPT, num_inference_steps=25, guidance_scale=7.5).images[0]
        image.save(os.path.join(CLASS_DIR, f"class_{len(existing_class) + i}.png"))
        if (i + 1) % 10 == 0:
            print(f"  {i + 1}/{num_to_generate}")

    del pipe_tmp
    torch.mps.empty_cache()
    print("Done.")
else:
    print(f"Already have {len(existing_class)} class images.")

Already have 50 class images.


In [8]:
class DreamBoothDataset(Dataset):
    def __init__(self, instance_dir, class_dir, instance_prompt, class_prompt, size=512):
        self.instance_images = [p for p in Path(instance_dir).iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS]
        self.class_images = [p for p in Path(class_dir).iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS][:NUM_CLASS_IMAGES]
        self.instance_prompt = instance_prompt
        self.class_prompt = class_prompt
        self.num_instance = len(self.instance_images)
        self.num_class = len(self.class_images)
        self.length = max(self.num_instance, self.num_class)
        self.transform = transforms.Compose([
            transforms.Resize((size, size)),
            transforms.CenterCrop(size),
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5]),
        ])

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        inst_img = Image.open(self.instance_images[idx % self.num_instance]).convert("RGB")
        cls_img = Image.open(self.class_images[idx % self.num_class]).convert("RGB")
        return {
            "instance_pixels": self.transform(inst_img),
            "instance_prompt": self.instance_prompt,
            "class_pixels": self.transform(cls_img),
            "class_prompt": self.class_prompt,
        }

dataset = DreamBoothDataset(INSTANCE_DIR, CLASS_DIR, INSTANCE_PROMPT, CLASS_PROMPT)
print(f"Instance: {dataset.num_instance}, Class: {dataset.num_class}")

Instance: 10, Class: 30


In [9]:
tokenizer = CLIPTokenizer.from_pretrained(MODEL_ID, subfolder="tokenizer")
text_encoder = CLIPTextModel.from_pretrained(MODEL_ID, subfolder="text_encoder", torch_dtype=DTYPE).to(DEVICE)
vae = AutoencoderKL.from_pretrained(MODEL_ID, subfolder="vae", torch_dtype=DTYPE).to(DEVICE)
unet = UNet2DConditionModel.from_pretrained(MODEL_ID, subfolder="unet", torch_dtype=DTYPE).to(DEVICE)
noise_scheduler = DDPMScheduler.from_pretrained(MODEL_ID, subfolder="scheduler")

vae.requires_grad_(False)
text_encoder.requires_grad_(False)

lora_config = LoraConfig(
    r=4,
    lora_alpha=4,
    target_modules=["to_k", "to_q", "to_v", "to_out.0"],
    lora_dropout=0.0,
)

unet = get_peft_model(unet, lora_config)
unet.print_trainable_parameters()

You are using a model of type `clip_text_model` to instantiate a model of type `clip`. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.
Loading weights: 100%|█████████████████| 196/196 [00:00<00:00, 1077.99it/s]
CLIPTextModel LOAD REPORT from: runwayml/stable-diffusion-v1-5
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/Users/bogdan/Developer/environments/venv/torch-build/lib/python3.14/site-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and igno

trainable params: 797,184 || all params: 860,318,148 || trainable%: 0.0927


In [10]:
def generate_validation_image(unet_model, vae, text_encoder, tokenizer, noise_scheduler, prompt, num_steps=30):
    unet_model.eval()

    tokens = tokenizer(
        prompt, padding="max_length", truncation=True,
        max_length=tokenizer.model_max_length, return_tensors="pt"
    ).input_ids.to(DEVICE)

    with torch.no_grad():
        encoder_hidden_states = text_encoder(tokens)[0]

        latents = torch.randn(1, 4, 64, 64, device=DEVICE, dtype=DTYPE)
        noise_scheduler_inf = DDPMScheduler.from_pretrained(MODEL_ID, subfolder="scheduler")
        noise_scheduler_inf.set_timesteps(num_steps)

        for t in noise_scheduler_inf.timesteps:
            t_input = t.unsqueeze(0).to(DEVICE)
            noise_pred = unet_model(latents, t_input, encoder_hidden_states).sample
            latents = noise_scheduler_inf.step(noise_pred, t, latents).prev_sample

        latents = latents / vae.config.scaling_factor
        image = vae.decode(latents).sample
        image = (image / 2 + 0.5).clamp(0, 1)
        image = image.cpu().permute(0, 2, 3, 1).float().numpy()[0]
        image = Image.fromarray((image * 255).astype("uint8"))

    unet_model.train()
    return image

In [11]:
NUM_EPOCHS = 400
LR = 5e-5
PRIOR_LOSS_WEIGHT = 1.0
VAL_EVERY = 20

dataloader = DataLoader(dataset, batch_size=1, shuffle=True)
optimizer = torch.optim.AdamW(unet.parameters(), lr=LR, weight_decay=1e-2)

unet.train()
val_images = []

for epoch in tqdm(range(NUM_EPOCHS), desc="Training"):
    total_loss = 0

    for batch in dataloader:
        inst_pixels = batch["instance_pixels"].to(DEVICE, dtype=DTYPE)
        cls_pixels = batch["class_pixels"].to(DEVICE, dtype=DTYPE)

        inst_tokens = tokenizer(
            batch["instance_prompt"], padding="max_length", truncation=True,
            max_length=tokenizer.model_max_length, return_tensors="pt"
        ).input_ids.to(DEVICE)

        cls_tokens = tokenizer(
            batch["class_prompt"], padding="max_length", truncation=True,
            max_length=tokenizer.model_max_length, return_tensors="pt"
        ).input_ids.to(DEVICE)

        with torch.no_grad():
            inst_enc = text_encoder(inst_tokens)[0]
            cls_enc = text_encoder(cls_tokens)[0]
            inst_latents = vae.encode(inst_pixels).latent_dist.sample() * vae.config.scaling_factor
            cls_latents = vae.encode(cls_pixels).latent_dist.sample() * vae.config.scaling_factor

        noise_i = torch.randn_like(inst_latents)
        t_i = torch.randint(0, noise_scheduler.config.num_train_timesteps, (1,), device=DEVICE).long()
        noisy_i = noise_scheduler.add_noise(inst_latents, noise_i, t_i)
        pred_i = unet(noisy_i, t_i, inst_enc).sample
        instance_loss = F.mse_loss(pred_i.float(), noise_i.float())

        noise_c = torch.randn_like(cls_latents)
        t_c = torch.randint(0, noise_scheduler.config.num_train_timesteps, (1,), device=DEVICE).long()
        noisy_c = noise_scheduler.add_noise(cls_latents, noise_c, t_c)
        pred_c = unet(noisy_c, t_c, cls_enc).sample
        prior_loss = F.mse_loss(pred_c.float(), noise_c.float())

        loss = instance_loss + PRIOR_LOSS_WEIGHT * prior_loss

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(unet.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(dataloader)

    if (epoch + 1) % VAL_EVERY == 0:
        tqdm.write(f"Epoch {epoch+1}/{NUM_EPOCHS}, Loss: {avg_loss:.4f}")
        val_img = generate_validation_image(unet, vae, text_encoder, tokenizer, noise_scheduler, VAL_PROMPT)
        val_img.save(os.path.join(VAL_DIR, f"val_epoch_{epoch+1}.png"))
        val_images.append((epoch + 1, val_img))
        tqdm.write(f"  Validation image saved.")

print("Training complete!")

Training:   5%|█▏                       | 19/400 [18:53<5:55:23, 55.97s/it]

Epoch 20/400, Loss: 0.2610


Training:   5%|█▎                       | 20/400 [19:05<6:17:23, 59.59s/it]

  Validation image saved.


Training:  10%|██▍                      | 39/400 [38:24<5:50:37, 58.28s/it]

...

Training:  75%|████████████████▍     | 299/400 [5:16:31<1:41:16, 60.16s/it]

Epoch 300/400, Loss: 0.2188


Training:  75%|████████████████▌     | 300/400 [5:16:45<1:51:07, 66.68s/it]

  Validation image saved.


Training:  76%|████████████████▋     | 304/400 [5:20:59<1:41:21, 63.35s/it]

KeyboardInterrupt



In [12]:
unet.save_pretrained(OUTPUT_DIR)
print(f"LoRA weights saved to {OUTPUT_DIR}")

LoRA weights saved to ./lora_model


In [13]:
del unet, vae, text_encoder, tokenizer, optimizer, dataloader, dataset
torch.mps.empty_cache()

pipe = StableDiffusionPipeline.from_pretrained(MODEL_ID, torch_dtype=DTYPE)
pipe = pipe.to(DEVICE)
pipe.unet = PeftModel.from_pretrained(pipe.unet, OUTPUT_DIR)
pipe.safety_checker = None

NUM_STEPS_INF = 50
GUIDANCE_SCALE = 7.5

Loading pipeline components...:  29%|██▎     | 2/7 [00:01<00:04,  1.07it/s]You are using a model of type `clip_text_model` to instantiate a model of type `clip`. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.

Loading weights: 100%|██████████████████| 196/196 [00:00<00:00, 940.64it/s]
CLIPTextModel LOAD REPORT from: /Users/bogdan/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weig

In [ ]:
stylized_prompts = [
    "a portrait of sks person, face close-up",
    "a portrait of sks person in foreground, Chinese on background, sks person face close-up",
    "a portrait of sks person in foreground, Japanese city on background, face close-up",
    "a portrait of sks person, night, moon on background, face close-up",
    "a portrait of sks person in foreground, mountains on background, face close-up",
    "a portrait of sks person in a forest, hight quality, realism,, face close-up",
    "a portrait of sks person in a city, hight quality, realism, face close-up",
    "a portrait of sks person in a beach, hight quality, realism, face close-up",
]

In [41]:
stylized_images = []

for i, prompt in enumerate(stylized_prompts):
    image = pipe(
        prompt,
        num_inference_steps=NUM_STEPS,
        guidance_scale=GUIDANCE_SCALE,
    ).images[0]
    save_path = os.path.join(GENERATED_DIR, f"stylized_{i+50}.png")
    image.save(save_path)
    stylized_images.append(image)
    print(f"Generated: {save_path}")

100%|██████████████████████████████████████| 50/50 [00:32<00:00,  1.56it/s]


Generated: stylized_1.png from a photo of sks man and Vladimir Putin, face close-up


In [ ]:
test_prompts = [
    "a portrait of sks person, spider man superhero costume, adult, stubble on face, no mask, face close-up"
]

for i, prompt in enumerate(test_prompts):
    image = pipe(
        prompt,
        num_inference_steps=NUM_STEPS,
        guidance_scale=GUIDANCE_SCALE,
    ).images[0]
    save_path = os.path.join(GENERATED_DIR, f"stylized_{i+30}.png")
    image.save(save_path)
    print(f"Generated: {save_path}")

In [ ]:
standard_prompts = [
    "man in a forest, high quality, realism",
    "man in a city, high quality, realism",
    "man on a beach, high quality, realism",
]

standard_images = []
for i, prompt in enumerate(standard_prompts):
    image = pipe(prompt, num_inference_steps=NUM_STEPS, guidance_scale=GUIDANCE_SCALE).images[0]
    image.save(os.path.join(GENERATED_DIR, f"standard_{i+1}.png"))
    standard_images.append(image)

In [ ]:
import open_clip

clip_model, _, clip_preprocess = open_clip.create_model_and_transforms("ViT-B-32", pretrained="openai")
clip_model = clip_model.to(DEVICE)
clip_tokenizer_fn = open_clip.get_tokenizer("ViT-B-32")

def compute_clip_score(image, text):
    img_tensor = clip_preprocess(image).unsqueeze(0).to(DEVICE)
    text_tokens = clip_tokenizer_fn([text]).to(DEVICE)
    with torch.no_grad():
        img_f = clip_model.encode_image(img_tensor)
        txt_f = clip_model.encode_text(text_tokens)
        img_f = img_f / img_f.norm(dim=-1, keepdim=True)
        txt_f = txt_f / txt_f.norm(dim=-1, keepdim=True)
    return (img_f @ txt_f.T).item()

all_prompts = stylized_prompts + standard_prompts
all_images = stylized_images + standard_images

print("CLIP Scores")
clip_scores = []
for img, prompt in zip(all_images, all_prompts):
    score = compute_clip_score(img, prompt)
    clip_scores.append(score)
    print(f"  {prompt}: {score:.4f}")
print(f"  Average: {sum(clip_scores)/len(clip_scores):.4f}")

In [ ]:
import lpips

lpips_fn = lpips.LPIPS(net="alex").to(DEVICE)
lpips_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3),
])

print("LPIPS")
lpips_scores = []
for i in range(len(stylized_images)):
    for j in range(i + 1, len(stylized_images)):
        img1 = lpips_transform(stylized_images[i]).unsqueeze(0).to(DEVICE)
        img2 = lpips_transform(stylized_images[j]).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            d = lpips_fn(img1, img2).item()
        lpips_scores.append(d)
        print(f"  {stylized_prompts[i]} vs {stylized_prompts[j]}: {d:.4f}")
print(f"  Average: {sum(lpips_scores)/len(lpips_scores):.4f}")

In [ ]:
import numpy as np
import cv2

def laplacian_sharpness(pil_image):
    img_np = np.array(pil_image.convert("L"))
    return cv2.Laplacian(img_np, cv2.CV_64F).var()

print("Laplacian Sharpness")
sharpness_scores = []
for img, prompt in zip(all_images, all_prompts):
    score = laplacian_sharpness(img)
    sharpness_scores.append(score)
    print(f"  {prompt}: {score:.2f}")
print(f"  Average: {sum(sharpness_scores)/len(sharpness_scores):.2f}")